In [1]:
import numpy as np 
import pandas as pd
import scanpy as sc
import mudata 
import matplotlib.pyplot as plt
import seaborn as sns

# Description:

This notebook is used to perform GRN analysis.

In [ ]:
e_direct = pd.read_csv('/home/rsun@ZHANGroup.local/sly_data/sly_07_exfig/global_grn/scplus_pipeline/Snakemake/eRegulon_direct.tsv', sep='\t')
e_direct.loc[:,'eRegulon_name'].unique().shape

In [ ]:
auc_direct = mudata.read('/home/rsun@ZHANGroup.local/sly_data/sly_07_exfig/global_grn/scplus_pipeline/Snakemake/AUCell_direct.h5mu')
auc_direct

In [ ]:
cell_data = pd.read_csv('/home/rsun@ZHANGroup.local/sly_data/sly_07_exfig/global_grn/scenic_brain_out/cell_data.csv', index_col = 0)
cell_data

In [ ]:
scdata = auc_direct['Gene_based'] 
scdata.obs = cell_data.loc[scdata.obs.index,:]
scdata.obs

## get TF expression data

In [18]:
exp_data = sc.read_h5ad('/home/rsun@ZHANGroup.local/cross_embed/data/species_raw.h5ad')
new_idx = []
for ele in exp_data.obs.index:
    a, b, c = ele.split('-')
    new_idx.append(b + '-' + c +  '_' + a)
exp_data.obs.index = new_idx 

exp_data = exp_data[scdata.obs.index,:]
assert (exp_data.obs.index == scdata.obs.index).all()

## Only save eRegulon with '+/+' and '-/-

In [8]:
module_list = []
sel_idx = []
for ele in scdata.var.index:
    a,b,c,d = ele.split('_')

    if c == '+/+':
        module_list.append(f'{a}(+)')
        sel_idx.append(True)
    elif c == '-/-':
        module_list.append(f'{a}(-)')
        sel_idx.append(True)
    else:
        sel_idx.append(False)
print(len(module_list)) 

scdata = scdata[:,sel_idx]
scdata.var.index = module_list 
#scdata.var

77


/tmp/ipykernel_1703972/846938510.py:17: ImplicitModificationWarning: Trying to modify index of attribute `.var` of view, initializing view as actual.
  scdata.var.index = module_list


In [9]:
from sklearn.preprocessing import StandardScaler
def _scale(X: pd.DataFrame) -> pd.DataFrame:
    return (X - X.min()) / (X.max() - X.min())

def z_scale(df):
    scaler = StandardScaler()
    df = pd.DataFrame(scaler.fit_transform(df), columns=df.columns, index = df.index)
    return df

def sum_scale(df):
    new_df = df / df.sum(axis = 0).values
    return new_df

In [10]:
import os 
save_dir = 'reg_plot'
os.makedirs(save_dir, exist_ok=True)

## Group the regulation based on species category 

generate a dataframe with size (species, eRegulation)

In [ ]:
reg_dic = {}
for key in scdata.obs.species.unique():
    idx = scdata.obs.species == key
    tmp = scdata[idx,:].X.mean(axis = 0)
    reg_dic[key] = tmp 
reg_df = pd.DataFrame(reg_dic)

reg_df = reg_df.T
reg_df.columns = scdata.var.index
reg_df= reg_df.loc[['CY','JT','T','M','QF'],:]
reg_df = sum_scale(reg_df)

plt.figure(figsize = (30,4))
sns.heatmap(reg_df, cmap = 'RdYlBu_r')#, vmax = 0.5)
plt.title('brain regulation among 5 species')
plt.savefig(os.path.join(save_dir,'brain_regulation_5_species.png'), bbox_inches = 'tight')
plt.show()

## Group the regulation based on the celltype

generate a dataframe with size (celltype, eRegulon)

In [ ]:
reg_dic = {}
for key in scdata.obs.celltype.unique():
    idx = scdata.obs.celltype == key
    tmp = scdata[idx,:].X.mean(axis = 0)
    reg_dic[key] = tmp 
reg_df = pd.DataFrame(reg_dic)


reg_df = reg_df.T
reg_df.columns = scdata.var.index
reg_df = sum_scale(reg_df)

plt.figure(figsize = (30,7))
sns.heatmap(reg_df, cmap = 'RdYlBu_r',vmax = 0.4)
plt.title('brain regulation among cell types')
plt.savefig(os.path.join(save_dir,'brain_regulation cell type.png'), bbox_inches = 'tight')
plt.show()

## Group the regulation based on the celltype category for each species 

generate 5 dataframe(5 speices), each df with shape (celltype, eRegulon)

In [12]:
save_dir = 'figures/speices_celltype'
os.makedirs(save_dir, exist_ok=True)

In [ ]:
reg_dic = {}
for key in scdata.obs.species.unique():
    species_res = {}
    idx = scdata.obs.species == key
    subdata = scdata[idx,:]
    
    for ele in subdata.obs.celltype.unique():
        idx2 = subdata.obs.celltype == ele
        tmp = subdata[idx2,:].X.mean(axis = 0)
        species_res[ele] = tmp
    reg_dic[key] = species_res


    reg_df = pd.DataFrame(species_res)

    reg_df = reg_df.T
    reg_df.columns = scdata.var.index
    reg_df= reg_df.loc[scdata.obs.celltype.unique(),:]
    reg_df = sum_scale(reg_df)

    plt.figure(figsize = (30,4))
    sns.heatmap(reg_df, cmap = 'RdYlBu_r', vmax = 0.5)
    plt.title(f'{key} brain regulation among 7 celltype')
    plt.savefig(os.path.join(save_dir,f'{key}_brain_regulation.png'), bbox_inches = 'tight')
    plt.show()

In [ ]:
sel_tf = ['Dlx1(+)','Dlx2(+)','Arx(+)','Junb(+)','Neurod1(+)','Uncx(+)','Zic2(+)','Sox10(+)','Sox2(+)','Sox8(+)','Pgr(+)','Rarb(+)',
          'Pou6f1(+)','Tfap2a(+)','Bach1(+)','Klf4(+)','Esr1(+)','Six1(+)','Lef1(+)','Prdm5(+)','Sox6(+)','Sox9(+)','Elk3(+)',
          'Etv6(+)','Fli1(+)','Irf5(+)','Irf8(+)']

reg_dic = {}
for key in scdata.obs.species.unique():
    species_res = {}
    idx = scdata.obs.species == key
    subdata = scdata[idx,:]
    
    for ele in subdata.obs.celltype.unique():
        idx2 = subdata.obs.celltype == ele
        tmp = subdata[idx2,:].X.mean(axis = 0)
        species_res[ele] = tmp
    reg_dic[key] = species_res


    reg_df = pd.DataFrame(species_res)

    reg_df = reg_df.T
    reg_df.columns = scdata.var.index
    reg_df = reg_df.loc[:,sel_tf]
    reg_df= reg_df.loc[scdata.obs.celltype.unique(),:]
    reg_df = sum_scale(reg_df)

    plt.figure(figsize = (30,4))
    sns.heatmap(reg_df, cmap = 'RdYlBu_r', vmax = 0.5)
    plt.title(f'{key} brain regulation among 7 celltype')
    plt.savefig(os.path.join(save_dir,f'{key}_brain_regulation_filter.pdf'), bbox_inches = 'tight')
    plt.show() 

    reg_df.to_csv(os.path.join(save_dir, f'{key}_regulation.csv'))